# Notebook 6 — an atom from first principles

Level-1 code: five levels, Einstein coefficients, branching $P(u\to l) = A_{ul}/\sum_k A_{uk}$, and explicit cascades that emit one photon per step.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[0] / "src"))
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import rtedu
from rtedu import results
from rtedu.visualization import save_fig, OI
from rtedu.atom import five_level_atom, H_ERG_S
rng = np.random.default_rng(rtedu.SEEDS["ch06"])
atom = five_level_atom()
T, n_total, t = 4000.0, 30.0, 2.0 * rtedu.DAY        # the book's standard state from here on
r_out = 0.2 * rtedu.C * t
tau = atom.line_list(T, n_total, t); emis = atom.thermal_emissivity(T, n_total)
nm = 1e7 * atom.lam_cm

In [ ]:
levels_cm = [0.0, 4000.0, 9000.0, 15000.0, 22000.0]
A = {(4, 0): 2e7, (4, 1): 5e7, (4, 2): 1e8, (4, 3): 3e6, (3, 0): 4e7, (3, 1): 2e7, (3, 2): 8e6, (2, 0): 6e7, (2, 1): 1e7, (1, 0): 5e6}

def branching(level):
    lines = [(u, l) for (u, l) in A if u == level]
    tot = sum(A[x] for x in lines)
    return lines, [A[x] / tot for x in lines]

def cascade(rng, level):
    photons = []
    while level > 0:
        lines, p = branching(level)
        u, l = lines[rng.choice(len(lines), p=p)]
        photons.append((u, l, levels_cm[u] - levels_cm[l]))         # energy in cm^-1
        level = l
    return photons

for level in range(4, 0, -1):
    lines, p = branching(level)
    print(f"from E{level}:", ", ".join(f"->E{l} {pp:.3f}" for (u, l), pp in zip(lines, p)))

In [ ]:
n = 20_000
E4 = levels_cm[4]
routes = {}; energy_per_line = {}
for _ in range(n):
    c = cascade(rng, 4)
    key = "->".join(str(u) for (u, l, e) in c) + "->0"
    routes[key] = routes.get(key, 0) + 1
    assert abs(sum(e for (_, _, e) in c) - E4) < 1e-9          # energy conserved by construction
    for (u, l, e) in c:
        energy_per_line[(u, l)] = energy_per_line.get((u, l), 0.0) + e / n
routes = dict(sorted(routes.items(), key=lambda kv: -kv[1]))
for k, v in list(routes.items())[:6]:
    print(f"route {k:>14s}: {v / n:.3f}")
mean_photons = sum(len(k.split("->")) - 1 for k in routes) / len(routes)

## Validation against `rtedu`

In [ ]:
assert np.allclose([atom.branching(l)[1] for l in (4,)][0], branching(4)[1])
c_ref = atom.cascade(np.random.default_rng(3), 4)
assert atom.lower[c_ref[-1]] == 0
e_line = np.array([energy_per_line.get((int(atom.upper[k]), int(atom.lower[k])), 0.0) for k in range(atom.n_lines)])
print("energy per line per cascade (cm^-1):", np.round(e_line, 1), " total", e_line.sum())

In [ ]:
from rtedu.visualization import level_diagram
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
level_diagram(axes[0], atom); axes[0].set_title("the five-level toy atom and its ten lines [nm]", fontsize=9)
axes[1].bar(np.arange(atom.n_lines), e_line / E4, color=OI["orange"]); axes[1].set_xticks(np.arange(atom.n_lines)); axes[1].set_xticklabels([f"{v:.0f}" for v in nm], rotation=60, fontsize=7)
axes[1].set_ylabel("fraction of the absorbed energy"); axes[1].set_title("where the energy of an absorption into $E_4$ comes out", fontsize=9)
fig.tight_layout(); save_fig(fig, "ch06_atom")

In [ ]:
results.record("ch06", dict(levels_cm=levels_cm, n_lines=atom.n_lines, lines_nm=nm, A=[atom.A[k] for k in range(atom.n_lines)],
                            branching_from_4=branching(4)[1], n_cascades=n, top_routes={k: v / n for k, v in list(routes.items())[:6]},
                            n_routes=len(routes), energy_fraction_per_line=e_line / E4,
                            mean_photons_per_cascade=float(np.mean([len(k.split("->")) - 1 for k in np.repeat(list(routes), list(routes.values()))]))))